# Pre-process CellChat objects

In [1]:
library_load <- suppressMessages(
    
    list(
        
        # CellTalk
        library(CellChat), 
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        
        # Plotting 
        library(ComplexHeatmap), 
        library(circlize), 
        library(viridis), 
        library(ggplotify), 
        library(ggrepel), 
        library(cowplot), 
            
        # Pyhton compatibility
        library(reticulate)
        
    )
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
ht_opt$message=FALSE # ComplexHeatmap 

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Source files 
source("plotting_global.R")

In [ ]:
# Plotting Theme
ggplot2::theme_set(theme_global_set()) # From project global source()

# Parameter and files 

In [ ]:
so_file <- "data/object/pp.rds"

# Import 

In [ ]:
so <- readRDS(so_file)

# Run CellChat pre-processing 

In [ ]:
# Pre-process cellchat objects for each treatment condition 
cellchat_pp <- function(so, group_by, file) {
    
    # Normalized expression matrix 
    so <- NormalizeData(so)
    norm <- GetAssayData(so, assay="RNA", slot="data")
    
    # Get meta data 
    meta <- so@meta.data
    meta <- droplevels(meta)
    
    # Run CellChat pipeline
    cellchat <- createCellChat(object=norm, meta=meta, group.by=group_by)
    cellchat <- setIdent(cellchat, ident.use=group_by)
    cellchat@DB <- CellChatDB.human # Add DB slot with interaction, complex, cofactor, and geneInfo data frames 
    cellchat <- subsetData(cellchat) # Subset data for signaling genes 
    # cellchat <- identifyOverExpressedGenes(cellchat, thresh.p=1) # Identify DE signaling genes stored in object@var.features. 
    cellchat@var.features <- list(features=rownames(cellchat@data))
    cellchat <- identifyOverExpressedInteractions(cellchat) # Filter interaction based on over expressed genes. Adds @LR$LRsig which is a pruned persion of cellchat@DB$interaction
    cellchat <- smoothData(cellchat, adj=PPI.human) # Project protein data 
    cellchat <- computeCommunProb(cellchat, raw.use=FALSE, population.size=TRUE, type="triMean") # Using the information from DB compute interaction strength (probability) in object@net$prob and associated p-values in object@net$pval. 
    cellchat <- computeCommunProbPathway(cellchat) # Summarise L-R interaction on pathway level. Return object@netP$prob is the pathway probability and significant pathways are stored in object@netP$pathways
    cellchat <- aggregateNet(cellchat) # Updates object@net with counts and weights 
    cellchat <- netAnalysis_computeCentrality(cellchat)
    
    # Save result 
    saveRDS(cellchat, file)
    
}

# Run CellChat for Skin data with celltype low label 

In [ ]:
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Baseline"), "celltype_low", "data/cellchat/skin_baseline_celltype_low.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Tx"), "celltype_low", "data/cellchat/skin_tx_celltype_low.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D14"), "celltype_low", "data/cellchat/skin_d14_celltype_low.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D100"), "celltype_low", "data/cellchat/skin_d100_celltype_low.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="GVHD"), "celltype_low", "data/cellchat/skin_gvhd_celltype_low.rds")

# Run CellChat for Skin data with celltype high label 

In [ ]:
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Baseline"), "celltype_high", "data/cellchat/skin_baseline_celltype_high.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="Tx"), "celltype_high", "data/cellchat/skin_tx_celltype_high.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D14"), "celltype_high", "data/cellchat/skin_d14_celltype_high.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="D100"), "celltype_high", "data/cellchat/skin_d100_celltype_high.rds")
cellchat_pp(subset(so, subset=hto_demux_class=="Skin" & time_point=="GVHD"), "celltype_high", "data/cellchat/skin_gvhd_celltype_high.rds")

# Session Info 

In [ ]:
sessionInfo()